# MASSDOT DATA COLLECTION
# (Expected Runtime: 2 minutes)

If unable to run this notebook, datasets can be found in this google drive folder:
https://drive.google.com/file/d/1Lm2-fgR1jxtbn4YT3_4mRejZJ4K0C_F8/view?usp=drive_link

# Importing all libraries

In [1]:
# !pip install googlemaps
# uncomment the line above if we decide to pay for GOOGLE CLOUD, an estimate of the cost is below $200
# GOOGLE CLOUD can be used to geocode 16911 crashes that have addresses but not lon and lat
# more information on how to do this can be found here: https://tinyurl.com/mr42hkxw
!pip install pandas
!pip install numpy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Collecting all Crash Data from MASSDOT, starting from 2022

In [2]:
all_features = []

#EDIT HERE TO CHANGE YEARS INCLUDED
years = range(2022, 2027)

#EDIT HERE TO EDIT PLACES INCLUDED
cities = ['BOSTON', 'CAMBRIDGE', 'SOMERVILLE', 'BROOKLINE']
city_str = ", ".join([f"'{c}'" for c in cities])

for year in years:
   
    base_url = f"https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT/MASSDOT_ODP_OPEN_{year}/FeatureServer/0/query"
    
    params = {
        "where": f"CITY_TOWN_NAME IN ({city_str})",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "returnGeometry": "true",
        "resultOffset": 0,
        "resultRecordCount": 2000
    }

    while True:
        
        response = requests.get(base_url, params=params)
        data = response.json()
        features = data.get("features", [])
        
        if not features:
            break
        
        all_features.extend(features)
        params["resultOffset"] += params["resultRecordCount"]

records = [f["attributes"] for f in all_features]

df = pd.DataFrame(records)
print("Shape:", df.shape)
df.head()

Shape: (27377, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATE_TEXT,CRASH_TIME_2,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,5054461,BOSTON,01 02 2022,1:39 AM,1641087540000,01:00AM to 01:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Principal Arterial - Other,4744161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5054385,BOSTON,01 01 2022,8:09 PM,1641067740000,08:00PM to 08:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Interstate,4744237,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5054357,BOSTON,01 03 2022,6:29 PM,1641234540000,06:00PM to 06:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Principal Arterial - Other,4744265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5054222,BOSTON,01 01 2022,8:45 AM,1641026700000,08:00AM to 08:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),1,...,NaN,Principal Arterial - Other Freeways or Express...,4744400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5054212,BOSTON,01 01 2022,6:12 PM,1641060720000,06:00PM to 06:59PM,Closed,Non-fatal injury,Suspected Minor Injury (B),1,...,NaN,Interstate,4744410,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
mass_crashes = df
print("Shape of MASSDOT dataset:", mass_crashes.shape)
mass_crashes.head()

Shape of MASSDOT dataset: (27377, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL,CRASH_TIME,CRASH_DATE
0,5054461,BOSTON,1641087540000,01:00AM to 01:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4744161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01:39:00,2022-01-02
1,5054385,BOSTON,1641067740000,08:00PM to 08:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4744237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20:09:00,2022-01-01
2,5054357,BOSTON,1641234540000,06:00PM to 06:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4744265,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18:29:00,2022-01-03
3,5054222,BOSTON,1641026700000,08:00AM to 08:59AM,Closed,Property damage only (none injured),No Apparent Injury (O),1,0,0,...,4744400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08:45:00,2022-01-01
4,5054212,BOSTON,1641060720000,06:00PM to 06:59PM,Closed,Non-fatal injury,Suspected Minor Injury (B),1,1,0,...,4744410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18:12:00,2022-01-01


# Standardizing dataset

**The following code creates new columnss:**
* The CYCLIST column has a value of 1 if a cyclist was involved in a crash, 0 otherwise.
* The PEDESTRIAN column has a value of 1 if a pedestrian was involved in a crash, 0 otherwise.
* The OTHER column has a value of 1 if another vulnerable user was involved in a crash, 0 otherwise.

**Standardizing MASSDOT crashes**

**Terms identified as a Pedestrian:** 
* Pedestrian
* Non-Motorized Wheelchair User
* Roller Skater

**Terms identified as a Cyclist:** 
* Cyclist 
* Bicyclist
* Bicycle

**Terms identified as Other:** 
* moped
* other
* unknown
* skateboarder
* Motorized Scooter Rider
* passenger
* Emergency Responder
* Roadway Worker

In [4]:
col1 = mass_crashes["NON_MTRST_TYPE_CL"]
col2 = mass_crashes["MOST_HRMFL_EVT_CL"]

mass_crashes["CYCLIST"] = np.where(
    col1.str.contains("Cyclist|Bicyclist", case=False, na=False) |
    col2.str.contains("Cyclist|Bicyclist|Bicycle", case=False, na=False),
    1,
    0
)

mass_crashes["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Non-Motorized Wheelchair|Roller Skater", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

mass_crashes['INTERSTATE'] = np.where(
    mass_crashes['F_CLASS'].str.contains("Interstate", case=False, na=False),
    1,
    0
)

mass_crashes['SEVERITY'] = mass_crashes['CRASH_SEVERITY_DESCR'].map({
    'Fatal injury': 1,
    'Non-fatal injury': 2
}).fillna(0)

mass_crashes["OTHER"] = np.where(
    mass_crashes["NON_MTRST_TYPE_CL"].str.contains("moped|other|unknown|skateboarder|Motorized Scooter Rider|passenger|Emergency Responder|Roadway Worker", case=False, na=False) |
    mass_crashes["MOST_HRMFL_EVT_CL"].str.contains("moped|other vulnerable", case=False, na=False),
    1,
    0
)

mass_crashes['POLICE'] = mass_crashes['POLC_AGNCY_TYPE_DESCR']
mass_crashes['ID'] = mass_crashes['CRASH_NUMB']
mass_crashes['MUNI'] = mass_crashes['CITY_TOWN_NAME']
mass_crashes = mass_crashes.drop(columns=["CITY_TOWN_NAME", "CRASH_NUMB", "POLC_AGNCY_TYPE_DESCR"])
mass_crashes['SOURCE'] = 'MASSDOT'


**Columns are attributes of each individual crash:**

* SOURCE: specifies where the crash data comes from(MASSDOT, Vision Zero, Somerville
PD and Cambridge PD)
* ID: unique identifier of each crash. Cambridge PD did not have an ID attribute so one
was created by adding CPD_ to the index number of each crash.
* MUNI: specifies the area each crash occurred(Boston, Cambridge, Brookline,
Somerville)
* DATE: the date each crash occurred in this format YEAR-MONTH-DAY
* SEVERITY: it is 1 if the crash was fatal, 2 if the crash resulted in nonfatal injuries and
blank if neither
* CRASH_TIME: time each crash occured
* POLICE: specifies if Local, State, MBTA or Campus police reported the crash
* CYCLIST: 1 if cyclist was involved in the crash, 0 if not
* PEDESTRIAN: 1 if pedestrian was involved in the crash, 0 if not
* OTHER: 1 if other vulnerable user was involved in the crash, 0 if not
* LAT: latitude of the location where the crash occurred
* LON: longitude of the location where the crash occurred
* INTERSTATE: 1 if the crash occurred on an interstate highway, 0 if not

In [5]:
cols_to_move = [
    "SOURCE",
    
    "ID",

    "MUNI",

    "CRASH_DATE",

    "SEVERITY",

    "CRASH_TIME",

    "POLICE",

    "CYCLIST",

    "PEDESTRIAN",

    "OTHER",

    "LAT",

    "LON",

    "INTERSTATE"
    
]
df = mass_crashes[cols_to_move + [c for c in mass_crashes.columns if c not in cols_to_move]]
df.head()

,SOURCE,ID,MUNI,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,CYCLIST,PEDESTRIAN,OTHER,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,MASSDOT,5054461,BOSTON,2022-01-02,0.0,01:39:00,State police,0,0,0,...,NaN,Principal Arterial - Other,4744161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MASSDOT,5054385,BOSTON,2022-01-01,0.0,20:09:00,State police,0,0,0,...,NaN,Interstate,4744237,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MASSDOT,5054357,BOSTON,2022-01-03,0.0,18:29:00,State police,0,0,0,...,NaN,Principal Arterial - Other,4744265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MASSDOT,5054222,BOSTON,2022-01-01,0.0,08:45:00,State police,0,0,0,...,NaN,Principal Arterial - Other Freeways or Express...,4744400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MASSDOT,5054212,BOSTON,2022-01-01,2.0,18:12:00,State police,0,0,0,...,NaN,Interstate,4744410,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# We keep only relevant columns and drop rows that don't have a date
df_short = df.iloc[:, :13]
df_short = df_short.dropna(subset=["CRASH_DATE"])

In [7]:
# We make sure numerical data is of type int or float, and strings of type str
df_short["CRASH_DATE"] = pd.to_datetime(df_short["CRASH_DATE"]).dt.date
df_short['SEVERITY'] = pd.to_numeric(df_short['SEVERITY'], errors='coerce').astype('Int64')
df_short['INTERSTATE'] = df_short['INTERSTATE'].astype(str)
df_short['ID'] = df_short['ID'].astype(str)

In [8]:
df_short.info()

<class 'pandas.DataFrame'>
RangeIndex: 27377 entries, 0 to 27376
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SOURCE      27377 non-null  str    
 1   ID          27377 non-null  str    
 2   MUNI        27377 non-null  str    
 3   CRASH_DATE  27377 non-null  object 
 4   SEVERITY    27377 non-null  Int64  
 5   CRASH_TIME  27377 non-null  object 
 6   POLICE      27375 non-null  str    
 7   CYCLIST     27377 non-null  int64  
 8   PEDESTRIAN  27377 non-null  int64  
 9   OTHER       27377 non-null  int64  
 10  LAT         25257 non-null  float64
 11  LON         25257 non-null  float64
 12  INTERSTATE  27377 non-null  str    
dtypes: Int64(1), float64(2), int64(3), object(2), str(5)
memory usage: 3.6+ MB


In [9]:
df_short.head()

,SOURCE,ID,MUNI,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,CYCLIST,PEDESTRIAN,OTHER,LAT,LON,INTERSTATE
0,MASSDOT,5054461,BOSTON,2022-01-02,0,01:39:00,State police,0,0,0,42.301271,-71.048851,0
1,MASSDOT,5054385,BOSTON,2022-01-01,0,20:09:00,State police,0,0,0,42.283946,-71.043621,1
2,MASSDOT,5054357,BOSTON,2022-01-03,0,18:29:00,State police,0,0,0,NaN,NaN,0
3,MASSDOT,5054222,BOSTON,2022-01-01,0,08:45:00,State police,0,0,0,42.370579,-71.064546,0
4,MASSDOT,5054212,BOSTON,2022-01-01,2,18:12:00,State police,0,0,0,42.356923,-71.119906,1


### Renaming colunmns to align with BCU's jason script for the Crash Map

In [10]:
renaming = {
    "LAT": "lat",
    "LON": "lng",
    "CRASH_DATE": "date",
    "CRASH_TIME": "time",
    "CYCLIST": "cyclist",
    "PEDESTRIAN": "pedestrian",
    "OTHER": "other",
    "INTERSTATE": "interstate",
    "SEVERITY": "severity",
    "ID": "id",
    "MUNI": "muni",
    "SOURCE": "source",
    "POLICE": "police"
}

df = df_short.rename(columns= renaming)

In [11]:
df.to_csv("MASSDOT DATA.csv")